In [1]:
import sys
sys.path.append("../") # go to parent dir

from custom_helpers_py.utilities import camel_to_snake_case
import pandas as pd
from os import listdir
from os.path import join
import json

In [2]:
COMPRESSED_FOLDER_PATH = join("../.outFiles/analysis/compressed")

CT_SCRAPES_FOLDER_PATH = join("../.outFiles/capitol_trades/trade_scrapes")
file_list = listdir(CT_SCRAPES_FOLDER_PATH)
df_list = []
for file_name in file_list:
    if not file_name.endswith(".json"):
        continue
    with open(join(CT_SCRAPES_FOLDER_PATH, file_name), "r", encoding="utf-8") as in_file:
        tmp = json.loads(in_file.read())
    

    for_df_data_list = []
    data_list: list[dict] = tmp["dataList"]
    for obj in data_list:
        politician_obj = obj["politician"]
        politician_full_name = politician_obj["fullName"]

        issuer_obj = obj["issuer"]
        asset_name = issuer_obj["issuerName"]
        ticker = issuer_obj["ticker"]

        pub_date_obj = obj["pubDate"]
        notif_date = pub_date_obj["value"] + " " +  pub_date_obj["label"]

        tx_date_obj = obj["txDate"]
        tx_date = tx_date_obj["value"] + " " + tx_date_obj["label"]

        reporting_gap = obj["reportingGap"]
        owner = obj["owner"]
        tx_type = obj["txType"]
        amount = obj["value"].replace('–', "-")
        asset_price = obj["price"]

        to_add = {
            "p_full_name": politician_full_name,
            "asset_name": asset_name,
            "ticker": ticker,
            "notif_date": notif_date,
            "tx_date": tx_date,
            "owner": owner,
            "tx_type": tx_type,
            "amount": amount,
            "avg_ticker_price": asset_price,
            "data_source": "CAPITOL_TRADES"
        }
        for_df_data_list.append(to_add)
        
    df = pd.DataFrame(for_df_data_list)
    df_list.append(df)

master_df = pd.concat(df_list)

to_save_path = join(COMPRESSED_FOLDER_PATH, "ct.csv")
master_df.to_csv(to_save_path, index=False, header=True)

master_df


,p_full_name,asset_name,ticker,notif_date,tx_date,owner,tx_type,amount,avg_ticker_price,data_source
0,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Spouse,SELL,250K-500K,694.52,CAPITOL_TRADES
1,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Child,SELL,15K-50K,694.52,CAPITOL_TRADES
2,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Child,SELL,15K-50K,694.52,CAPITOL_TRADES
3,Dan Meuser,US TREASURY BILLS,N/A,06:05 Today,23 Feb 2024,Spouse,BUY,250K-500K,N/A,CAPITOL_TRADES
4,Nancy Pelosi,FORGE INVESTMENTS LLC,N/A,06:05 Today,4 Mar 2024,Spouse,BUY,1M-5M,N/A,CAPITOL_TRADES
...,...,...,...,...,...,...,...,...,...,...
85,Hal Rogers,BRIDGE BUILDER LARGE CAP VALUE FUND,BBVLX:US,2 Apr 2021,25 Mar 2021,Undisclosed,SELL,1K-15K,15.99,CAPITOL_TRADES
86,Hal Rogers,BRIDGE BUILDER SMALL MID CAP VALUE FUND,BBVSX:US,2 Apr 2021,25 Mar 2021,Undisclosed,SELL,1K-15K,14.34,CAPITOL_TRADES
87,Mark Green,American Airlines Group Inc,AAL:US,29 Mar 2021,24 Mar 2021,Joint,BUY,1K-15K,21.81,CAPITOL_TRADES
88,Mark Green,United Airlines Holdings Inc,UAL:US,29 Mar 2021,24 Mar 2021,Joint,BUY,1K-15K,53.83,CAPITOL_TRADES


In [3]:
# politicians df
POLITICIAN_DATA_FOLDER_PATH = join("../.outFiles/capitol_trades/politician_data")
file_list = listdir(POLITICIAN_DATA_FOLDER_PATH)

politician_data_list = []
for file_name in file_list:
    file_path = join(POLITICIAN_DATA_FOLDER_PATH, file_name)
    with open(file_path, "r", encoding="utf-8") as in_file: 
        file_obj: dict = json.loads(in_file.read())
    
    fact_dict:dict = file_obj["factDict"]
    if fact_dict.get("trade"):
        fact_dict["trades"] = "1"
        del fact_dict["trade"] 

    if fact_dict.get("issuer"):
        fact_dict["issuers"] = "1"
        del fact_dict["issuer"]
    
    to_add = {
        "id": file_name.removesuffix(".json"),
        "fullName": file_obj.get("fullName"),
        **file_obj.get("officeInfo"),
        **file_obj.get("factDict"),
        "committeeList": file_obj.get("committeeList")
    }

    to_add["yearsActive"] = to_add["yearsActive"].replace('–', "-")
    politician_data_list.append(to_add)

politician_df = pd.DataFrame(politician_data_list)
politician_df.columns = [camel_to_snake_case(col) for col in politician_df.columns]


def convert_num_str(in_str: str):
    if "K" in in_str:
        in_str = in_str.removesuffix("K")
        return int(float(in_str) * 1_000)
    
    if "M" in in_str:
        in_str = in_str.removesuffix("M")
        return int(float(in_str) * 1_000_000)

    if "," in in_str:
        in_str = in_str.replace(",","")
    return int(in_str)

politician_df["volume"] = politician_df["volume"].apply(convert_num_str)
politician_df["trades"] = politician_df["trades"].apply(convert_num_str)
politician_df["issuers"] = politician_df["issuers"].apply(convert_num_str)

def get_year_active_started(in_str: str):
    return int(in_str.split(" - ")[0])

def get_year_active_ended(in_str: str):
    to_return =  in_str.split(" - ")[1]
    if to_return == "current":
        to_return = 2025
    return int(to_return)

politician_df["year_active_started"] = politician_df["years_active"].apply(get_year_active_started)
politician_df["year_active_ended"] = politician_df["years_active"].apply(get_year_active_ended)

politician_df = politician_df.drop(columns=["id"])
politician_df["chamber"] = politician_df["chamber"].str.slice(0,1)
politician_df["party"] = politician_df["party"].str.slice(0,1)
politician_df["full_name_upper"] = politician_df["full_name"].str.upper()

to_save_path = join(COMPRESSED_FOLDER_PATH, "ct_politicians.csv")
politician_df.to_csv(to_save_path, index=False, header=True)

politician_df

,full_name,party,chamber,state,trades,issuers,volume,last_traded,district,years_active,date_of_birth,age,committee_list,year_active_started,year_active_ended,full_name_upper
0,Robert Aderholt,R,H,Alabama,2,2,16000,2022-12-05,4,1997 - 2025,1965-07-22,58,[Appropriations],1997,2025,ROBERT ADERHOLT
1,Jake Auchincloss,D,H,Massachusetts,1,1,33000,2022-03-22,4,2021 - 2025,1988-01-29,36,"[Transportation & Infrastructure, Strategic Co...",2021,2025,JAKE AUCHINCLOSS
2,Rick Allen,R,H,Georgia,70,19,4050000,2024-01-17,12,2015 - 2025,1951-11-07,72,"[Education & Labor, Energy & Commerce]",2015,2025,RICK ALLEN
3,Cindy Axne,D,H,Iowa,75,20,600000,2022-12-23,3,2019 - current,1965-04-20,58,[],2019,2025,CINDY AXNE
4,Earl Blumenauer,D,H,Oregon,198,86,3630000,2024-02-15,3,1995 - 2025,1948-08-16,75,"[Budget, Ways & Means]",1995,2025,EARL BLUMENAUER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,Steve Womack,R,H,Arkansas,1,1,33000,2022-11-17,3,2011 - 2025,1957-02-18,67,[Appropriations],2011,2025,STEVE WOMACK
209,Ann Wagner,R,H,Missouri,28,28,6000000,2023-11-30,2,2013 - 2025,1962-09-13,61,"[Financial Services, Foreign Affairs]",2013,2025,ANN WAGNER
210,Michael Waltz,R,H,Florida,2,1,65000,2021-06-16,6,2019 - 2025,1974-01-31,50,"[Intelligence, Armed Services, Foreign Affairs]",2019,2025,MICHAEL WALTZ
211,John Yarmuth,D,H,Kentucky,14,12,161000,2021-10-22,3,2007 - current,1947-11-04,76,[],2007,2025,JOHN YARMUTH
